<a href="https://colab.research.google.com/github/ArmandoBarrios/unidad-3-/blob/main/practica_5_unidad_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
#Carga el Dataset
from google.colab import drive
drive.mount('/content/drive')
#ID del archivo
#https:https:https://drive.google.com/file/d/1V5ANwclpm0o4aHH_HvFeD0P0db0rqZOO/view?usp=sharing
file_id = '1V5ANwclpm0o4aHH_HvFeD0P0db0rqZOO'
url =  f"https://drive.google.com/uc?id={file_id}"
# Importamos librerias necesarias para Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Uploading `prestamos_ok.csv`

To resolve the `FileNotFoundError`, please upload the `prestamos_ok.csv` file using the cell below. Once uploaded, you can re-run the cell `tSIs1pJ15y-C` to load the data.

In [12]:
prestamos_df = pd.read_csv(url)
prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


In [13]:

X = prestamos_df[['funded_amnt', "int_rate", "grade", 'purpose', 'addr_state',
'home_ownership', 'annual_inc', 'dti', 'revol_util',
'pub_rec_bankruptcies']]
# Variable objetivo o variable a predecir
y = prestamos_df['loan_status']

In [14]:
# Dividimos el dataFrame
# stratify es para que mantenga la misma proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size= 0.4, stratify=y )

In [15]:
# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((11944, 10), (7964, 10), (11944,), (7964,))

In [16]:
# Primero, debemos convertir las variables categóricas a numéricas usando One-Hot Encoding
# Aplicamos get_dummies a todo el DataFrame X para transformar las columnas de texto
X_encoded = pd.get_dummies(X, columns=['grade', 'purpose', 'addr_state', 'home_ownership'])

# Dividimos nuevamente los datos con el nuevo DataFrame codificado
X_train_enc, X_test_enc, y_train, y_test = train_test_split(X_encoded, y, random_state=42, test_size=0.4, stratify=y)

# Create Regressor with default properties
rfc1 = RandomForestClassifier(random_state=23)

# Fit estimator using encoded data
rfc1 = rfc1.fit(X_train_enc, y_train)

# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", rfc1.score(X_train_enc, y_train))

Precision del clasificador en fase de entrenamiento 1.0


In [17]:
# Realizar una prediccion con los datos de prueba usando el conjunto codificado
y_pred = rfc1.predict(X_test_enc)

# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte del clasificador Random Forest sin balanceo de clases : \n",
      classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))

print(f'\nMatriz de Confusion Random Forest sin balanceo de clases:\n', confusion_matrix(y_test, y_pred))



Reporte del clasificador Random Forest sin balanceo de clases : 
               precision    recall  f1-score   support

   No Pagado       0.42      0.03      0.05      1177
      Pagado       0.85      0.99      0.92      6787

    accuracy                           0.85      7964
   macro avg       0.64      0.51      0.49      7964
weighted avg       0.79      0.85      0.79      7964


Matriz de Confusion Random Forest sin balanceo de clases:
 [[  32 1145]
 [  44 6743]]


In [18]:
rfc2 = RandomForestClassifier(class_weight='balanced', random_state=23)
# Usamos los datos codificados (X_train_enc) en lugar de los originales
rfc2 = rfc2.fit(X_train_enc, y_train)
y_pred = rfc2.predict(X_test_enc)
print("Reporte del clasificador con balanceo de clases: \n", classification_report(y_test, y_pred,
target_names=["No Pagado", "Pagado"] ))
print('Matriz de Confusion con balanceo de clases \n' , confusion_matrix(y_test, y_pred))

Reporte del clasificador con balanceo de clases: 
               precision    recall  f1-score   support

   No Pagado       0.28      0.01      0.02      1177
      Pagado       0.85      1.00      0.92      6787

    accuracy                           0.85      7964
   macro avg       0.56      0.50      0.47      7964
weighted avg       0.77      0.85      0.79      7964

Matriz de Confusion con balanceo de clases 
 [[  11 1166]
 [  29 6758]]


In [19]:
rfc3 = RandomForestClassifier(n_estimators=200, max_depth=7, class_weight='balanced', random_state=23)
# Fit estimator using encoded data
rfc3 = rfc3.fit(X_train_enc, y_train)
y_pred = rfc3.predict(X_test_enc)
print("Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=7: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print('Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=7\n' ,
confusion_matrix(y_test, y_pred))


Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=7: 
               precision    recall  f1-score   support

   No Pagado       0.23      0.63      0.33      1177
      Pagado       0.91      0.62      0.74      6787

    accuracy                           0.62      7964
   macro avg       0.57      0.63      0.54      7964
weighted avg       0.81      0.62      0.68      7964

Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=7
 [[ 742  435]
 [2554 4233]]


In [20]:

rfc4 = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=23)
# Fit estimator using encoded data
rfc4 = rfc4.fit(X_train_enc, y_train)
y_pred = rfc4.predict(X_test_enc)
print("Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=10: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print('Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=10\n' ,
confusion_matrix(y_test, y_pred))

Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=10: 
               precision    recall  f1-score   support

   No Pagado       0.24      0.49      0.32      1177
      Pagado       0.89      0.74      0.81      6787

    accuracy                           0.70      7964
   macro avg       0.57      0.61      0.57      7964
weighted avg       0.80      0.70      0.74      7964

Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=10
 [[ 574  603]
 [1793 4994]]


In [21]:
# ==========================================
# Gráfica de radar - Random Forest (Plotly)
# ==========================================
import plotly.graph_objects as go
# --- Nombres de los modelos --
modelos = [
"RF sin balanceo",
"RF balanceado",
"RF bal. n=200, depth=7",
"RF bal. n=200, depth=10"
]
# --- Métricas reales --
accuracy = [0.85, 0.85, 0.65, 0.73]
recall_no_pagado = [0.03, 0.01, 0.58, 0.40]
recall_pagado = [0.99, 1.00, 0.66, 0.78]
f1_macro = [0.49, 0.47, 0.54, 0.57]
# --- Ejes --
metricas = ["Accuracy", "Recall No Pagado", "Recall Pagado", "F1 Macro"]
# --- Crear la figura ---
fig = go.Figure()
# Agregar cada modelo al radar
for i, modelo in enumerate(modelos):
    fig.add_trace(go.Scatterpolar(
        r=[accuracy[i], recall_no_pagado[i], recall_pagado[i], f1_macro[i], accuracy[i]],
        theta=metricas + [metricas[0]],
        fill='toself',
        name=modelo
    ))
# --- Personalización --
fig.update_layout(
    title="Comparación de modelos Random Forest (Radar Plot)",
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1], tickvals=[0.2, 0.4, 0.6, 0.8, 1.0]),
    ),
    showlegend=True
)
fig.update_layout(width=800, height=600)
fig.show()



In [ ]:
# Búsqueda de hiperparámetros con GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
# Iniciamos con un Modelo base con balanceo de clases (para compensar desbalance)
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
# --- Definimos el espacio de búsqueda --
param_grid = {
'n_estimators': [100, 200, 300],
'max_depth': [5, 7, 10 ],
'min_samples_split': [2, 5, 10],
'min_samples_leaf': [1, 2, 4],
'max_features': ['sqrt', 'log2']
}
# --- Configuración de GridSearchCV --
grid_search = GridSearchCV(
estimator=rf,
param_grid=param_grid,
scoring='f1_macro',
# Se puede cambiar a 'recall_macro', 'accuracy', etc.
cv=3,
n_jobs=-1,
verbose=2
)
# 3 particiones para validación cruzada
# usa todos los núcleos disponibles
# para ver el progreso
# --- Ejecutar búsqueda --
grid_search.fit(X_train_enc, y_train)
# --- Mostrar los mejores parámetros --
print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)
print("\n Mejor puntaje promedio (validación cruzada):")
print(grid_search.best_score_)
# --- Entrenar modelo final con los mejores parámetros --
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test_enc)
# --- Evaluar en conjunto de prueba --
print("\n Reporte de Clasificación (modelo óptimo):")
print(classification_report(y_test, y_pred_best, target_names=["No Pagado", "Pagado"]))
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred_best))

Fitting 3 folds for each of 162 candidates, totalling 486 fits


In [ ]:
from sklearn.metrics import make_scorer, recall_score
scorer = make_scorer(recall_score, pos_label=0)
grid_search = GridSearchCV(
RandomForestClassifier(class_weight='balanced', random_state=23),
param_grid=param_grid,
scoring=scorer,  # << prioriza recall de la clase No Pagado
cv=5,
n_jobs=-1,
verbose=2
)


In [ ]:
rom sklearn.metrics import make_scorer, fbeta_score
# F2 da más peso al recall (importante para detectar impagos)
scorer_f2 = make_scorer(fbeta_score, beta=2, pos_label=0)
grid_search = GridSearchCV(
RandomForestClassifier(class_weight='balanced', random_state=23),
param_grid=param_grid,
scoring=scorer_f2,
cv=5,
n_jobs=-1,
verbose=2
)

In [ ]:
from sklearn.metrics import make_scorer, recall_score, f1_score

grid_search = GridSearchCV(
RandomForestClassifier(class_weight='balanced', random_state=23),
param_grid=param_grid,
scoring={
'recall_no_pagado': make_scorer(recall_score, pos_label=0),
'f1_no_pagado': make_scorer(f1_score, pos_label=0),
'accuracy': 'accuracy'
},
refit='recall_no_pagado',  # <--- elige el modelo que maximice recall de No Pagado
cv=5,
n_jobs=-1,
verbose=2
)

In [ ]:
# Optimización de Random Forest priorizando detección de impagos
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score, classification_report, confusion_matrix
# Definición del modelo base con balanceo de clases
# El parámetro class_weight='balanced' ajusta el peso inversamente proporcional
# a la frecuencia de cada clase, ayudando a tratar el desbalance.
modelo_base = RandomForestClassifier(
class_weight='balanced',
random_state=23
)
# Definición de la cuadrícula de hiperparámetros
param_grid = {
'n_estimators': [100, 200, 300],
'max_depth': [5, 7, 10],
'min_samples_split': [2, 5, 10],
'min_samples_leaf': [1, 2, 4],
'max_features': ['sqrt', 'log2']
}
# Definición del criterio de evaluación
# Se usará el recall de la clase "No Pagado" (pos_label=0)
# Esto hace que el modelo busque detectar el mayor número posible de impagos.
scorer = make_scorer(recall_score, pos_label=0)
# Configuración del GridSearchCV
grid_search = GridSearchCV(
estimator=modelo_base,
param_grid=param_grid,
scoring=scorer,
# prioriza el recall de la clase No Pagado
cv=5,
n_jobs=-1,
verbose=2
)
# validación cruzada con 5 particiones
# usa todos los núcleos disponibles
# Entrenamiento del modelo con búsqueda de hiperparámetros
grid_search.fit(X_train_enc, y_train)
print("\nBúsqueda finalizada.")
print(f"Mejores hiperparámetros encontrados:\n{grid_search.best_params_}")
# Evaluación del mejor modelo
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_enc)
print("\nReporte de Clasificación (modelo optimizado para detectar impagos):\n")
print(classification_report(y_test, y_pred))
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))
